# OOD event-and-phase post-processing

This notebook uses the sibling `Data/` and `Result/` directories in the public
release. The forecasting checkpoints remain frozen. The default configuration
reads or creates public mean-10 priors and evaluates all four training
quantities for source OOD labels 2–5.

LLM generation is disabled by default. Enable it only when the saved public
priors are absent. Offline application and plotting never call an LLM.


In [ ]:
from pathlib import Path
import sys

CODE_DIR = Path.cwd()
if not (CODE_DIR / 'config.py').is_file():
    raise FileNotFoundError('Run this notebook with post_processing as the working directory.')
sys.path.insert(0, str(CODE_DIR))

import config as cfg

FINAL_ROOT = cfg.FINAL_ROOT
RUN_ROOT = None  # Set only if saved public outputs remain in another Result/OOD subfolder.
SAMPLES = 10
PROMPT_STYLE = 'public'
BUILDINGS = []
OODS = ['OOD-2', 'OOD-3', 'OOD-4', 'OOD-5']
TRAIN_DAYS = ['14-day', '21-day', '28-day', 'Full']
MAX_WORKERS = 6
ZONES_PER_CALL = 50
MAX_ATTEMPTS = 3
MODEL = cfg.MODEL
PROVIDER = 'together'
TEMPERATURE = 0.2
PRIOR_SOURCE = 'generated'
EXISTING_PRIORS = None
SAVE_ZONE_METRICS = False
TARGET_MATCHING = 'legacy'

RUN_GENERATION = False  # True calls the LLM; saved valid requests are reused.
RUN_APPLY = False       # True runs offline post-processing and evaluation.
FORCE_RECOMPUTE = False

S = cfg.settings(
    final_root=FINAL_ROOT, run_root=RUN_ROOT, samples=SAMPLES, prompt_style=PROMPT_STYLE,
    buildings=BUILDINGS, oods=OODS, train_days=TRAIN_DAYS,
    max_workers=MAX_WORKERS, zones_per_call=ZONES_PER_CALL,
    max_attempts=MAX_ATTEMPTS, provider=PROVIDER, model=MODEL,
    temperature=TEMPERATURE, prior_source=PRIOR_SOURCE,
    existing_priors=EXISTING_PRIORS, target_matching=TARGET_MATCHING,
    save_zone_metrics=SAVE_ZONE_METRICS,
)
print('Result root:', S['final_root'])
print('Case index:', S['case_index'])
print('Generation index:', S['input_index'])
print('Public output root:', cfg.generation_root(S))
print('Prompt / samples / metric convention:', PROMPT_STYLE, SAMPLES, TARGET_MATCHING)


## 1. Generate priors (optional)

The number of samples is the number of complete independent judgments per
building and OOD scenario. A local key can be supplied through the provider
environment variable or the ignored `api_key_local.txt` file. Keep generation
disabled when the public mean-10 priors are already present in `Result/`.


In [ ]:
if RUN_GENERATION:
    import generate_priors
    generate_priors.run(S, execute=True)
else:
    print('跳过LLM生成。需要只预览请求数，可单独运行下一行（无API）：')
    print('import generate_priors; generate_priors.preview(S)')


## 2. Apply gates and evaluate offline

Raw predictions, the complete fixed-gate grid, and the LLM event-and-phase
gate share the same deterministic correction candidate. RMSE and MAE use the
same paired cases and valid points. This step does not call an LLM.


In [ ]:
if RUN_APPLY:
    import apply_priors
    METRICS = apply_priors.run(S, force=FORCE_RECOMPUTE)
else:
    METRICS = cfg.metric_root(S)
    print('跳过离线应用；直接尝试读取已有指标：', METRICS)
    print('首次复现请把首格 RUN_APPLY 改为 True，再运行本节。')


## 3. Summaries and figures

Results are first averaged within each building and then equally across
buildings. Missing cases are reported and are never filled with zero.
`legacy` and `exact_air` evaluations are written separately.


In [ ]:
import visualize_results
RESULTS = visualize_results.report(S)


## 4. Coverage and architecture tables

This cell reads saved outputs only.


In [ ]:
import pandas as pd
from IPython.display import display

failure_path = METRICS / 'failures.csv'
if failure_path.is_file():
    try:
        failures = pd.read_csv(failure_path)
    except pd.errors.EmptyDataError:
        failures = pd.DataFrame()
    print('本次选择的未完成/失败case：', len(failures))
    if len(failures):
        display(failures.groupby(['Building', 'OOD_Label', 'Error_type']).size().rename('Cases').reset_index())

architecture_path = METRICS / 'figures/architecture_metrics.csv'
if architecture_path.is_file():
    architecture = pd.read_csv(architecture_path)
    architecture = architecture.loc[architecture.Postprocess_Method.isin(visualize_results.MAIN)]
    for day in TRAIN_DAYS:
        t = architecture.loc[architecture.Train_Days.eq(day)]
        if not t.empty:
            print('MAE /', day)
            display(t.pivot(index=['Architecture','OOD_Label'], columns='Postprocess_Method', values='MAE').reindex(columns=visualize_results.MAIN).round(4))
else:
    print('尚无架构表；完成离线应用和第3节后会生成。')
